# **Iterative_Feature_Engineering**

In [1]:
# --- Imports and Setup ---

# Data Manipulation
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 

# Utility for loading our old preprocessor
import joblib
import json
import warnings
warnings.filterwarnings('ignore')

# --- Pandas Display Options ---
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

print("✅ Imports and setup complete.")


✅ Imports and setup complete.


In [2]:
# --- Define File Paths ---
# Use the "Copy file path" button for each of these files from the Input sidebar.
# The paths might be slightly different for you.

TRAIN_PATH = '/kaggle/input/01-eda-and-baseline-ipynb/train_cleaned.csv'
TEST_PATH = '/kaggle/input/01-eda-and-baseline-ipynb/test_cleaned.csv'
PREPROCESSOR_PATH = '/kaggle/input/01-eda-and-baseline-ipynb/preprocessor_dense.joblib'
BEST_MODELS_PATH = '/kaggle/input/01-eda-and-baseline-ipynb/best_models_list.json'


# --- Load DataFrames ---
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
print("✅ Cleaned train and test data loaded.")

# --- Load Preprocessor ---
preprocessor = joblib.load(PREPROCESSOR_PATH)
print("✅ Preprocessor from Notebook 01 loaded.")

# --- Load Model Rankings ---
with open(BEST_MODELS_PATH, 'r') as f:
    best_models = json.load(f)
print("✅ Best models list from Notebook 01 loaded.")
print(f"Top performing model from baseline: {best_models[0]}")

# --- Display DataFrames to Verify ---
print("\n--- Training Data ---")
display(train_df.head())

print("\n--- Test Data ---")
display(test_df.head())


✅ Cleaned train and test data loaded.
✅ Preprocessor from Notebook 01 loaded.
✅ Best models list from Notebook 01 loaded.
Top performing model from baseline: XGBoost (Boosted)

--- Training Data ---


,annual_income,debt_to_income_ratio,credit_score,loan_amount,interest_rate,gender,marital_status,education_level,employment_status,loan_purpose,grade_subgrade,loan_paid_back
0,29367.99,0.084,736,2528.42,13.67,Female,Single,High School,Self-employed,Other,C3,1.0
1,22108.02,0.166,636,4593.10,12.92,Male,Married,Master's,Employed,Debt consolidation,D3,0.0
2,49566.20,0.097,694,17005.15,9.76,Male,Single,High School,Employed,Debt consolidation,C5,1.0
3,46858.25,0.065,533,4682.48,16.10,Female,Single,High School,Employed,Debt consolidation,F1,1.0
4,25496.70,0.053,665,12184.43,10.21,Male,Married,High School,Employed,Other,D1,1.0



--- Test Data ---


,id,annual_income,debt_to_income_ratio,credit_score,loan_amount,interest_rate,gender,marital_status,education_level,employment_status,loan_purpose,grade_subgrade
0,593994,28781.05,0.049,626,11461.42,14.73,Female,Single,High School,Employed,Other,D5
1,593995,46626.39,0.093,732,15492.25,12.85,Female,Married,Master's,Employed,Other,C1
2,593996,54954.89,0.367,611,3796.41,13.29,Male,Single,Bachelor's,Employed,Debt consolidation,D1
3,593997,25644.63,0.110,671,6574.30,9.57,Female,Single,Bachelor's,Employed,Debt consolidation,C3
4,593998,25169.64,0.081,688,17696.89,12.80,Female,Married,PhD,Employed,Business,C1


In [3]:
# --- Combine DataFrames for Safe Feature Engineering ---

print("Preparing data for feature engineering...")

# 1. Store the original test IDs for later use in submission, then drop the 'id' column.
# This prevents it from being accidentally used as a feature.
test_ids = test_df['id'].copy()
train_df_fe = train_df.copy()
test_df_fe = test_df.drop(columns=['id']).copy()

# 2. Add a 'source' column to each dataframe.
# This allows us to easily split them back apart after feature engineering is complete.
train_df_fe['source'] = 'train'
test_df_fe['source'] = 'test'

# 3. Concatenate (combine) the two dataframes into one.
df_combo = pd.concat([train_df_fe, test_df_fe], ignore_index=True)

# 4. Verify the process.
print("✅ Dataframes combined successfully.")
print(f"   - Shape of original training data: {train_df_fe.shape}")
print(f"   - Shape of original test data:     {test_df_fe.shape}")
print(f"   - Shape of combined dataframe:      {df_combo.shape}")

print("\n--- Sample of Combined Dataframe ---")
# Display the head and tail to show both train and test rows are present
display(df_combo.head())
display(df_combo.tail())

Preparing data for feature engineering...
✅ Dataframes combined successfully.
   - Shape of original training data: (593994, 13)
   - Shape of original test data:     (254569, 12)
   - Shape of combined dataframe:      (848563, 13)

--- Sample of Combined Dataframe ---


,annual_income,debt_to_income_ratio,credit_score,loan_amount,interest_rate,gender,marital_status,education_level,employment_status,loan_purpose,grade_subgrade,loan_paid_back,source
0,29367.99,0.084,736,2528.42,13.67,Female,Single,High School,Self-employed,Other,C3,1.0,train
1,22108.02,0.166,636,4593.10,12.92,Male,Married,Master's,Employed,Debt consolidation,D3,0.0,train
2,49566.20,0.097,694,17005.15,9.76,Male,Single,High School,Employed,Debt consolidation,C5,1.0,train
3,46858.25,0.065,533,4682.48,16.10,Female,Single,High School,Employed,Debt consolidation,F1,1.0,train
4,25496.70,0.053,665,12184.43,10.21,Male,Married,High School,Employed,Other,D1,1.0,train


,annual_income,debt_to_income_ratio,credit_score,loan_amount,interest_rate,gender,marital_status,education_level,employment_status,loan_purpose,grade_subgrade,loan_paid_back,source
848558,92835.97,0.068,744,29704.00,13.48,Female,Single,Bachelor's,Employed,Debt consolidation,B2,NaN,test
848559,48846.47,0.091,634,20284.33,9.58,Female,Married,High School,Employed,Debt consolidation,D4,NaN,test
848560,20668.52,0.096,718,26387.55,9.00,Male,Single,Master's,Employed,Debt consolidation,C4,NaN,test
848561,34105.09,0.094,739,11107.36,9.81,Male,Single,Bachelor's,Employed,Business,C2,NaN,test
848562,45627.53,0.118,624,19246.14,11.64,Female,Married,High School,Employed,Car,D3,NaN,test


In [4]:
test_ids

0         593994
1         593995
2         593996
3         593997
4         593998
           ...  
254564    848558
254565    848559
254566    848560
254567    848561
254568    848562
Name: id, Length: 254569, dtype: int64

In [5]:
# --- Feature Engineering ---

print("Creating Tier 1 features from 'employment_status'...")

# A list to hold the names of the new features we create in this cell.
tier1_features = []

# --- Feature 1: is_unemployed ---
# This flag identifies the highest-risk group (7.7% payback).
feature_name = 'is_unemployed'
df_combo[feature_name] = (df_combo['employment_status'] == 'Unemployed').astype(int)
tier1_features.append(feature_name)

# --- Feature 2: is_student ---
# This flag identifies the second-highest-risk group (26.3% payback).
feature_name = 'is_student'
df_combo[feature_name] = (df_combo['employment_status'] == 'Student').astype(int)
tier1_features.append(feature_name)

# --- Feature 3: is_retired ---
# This flag identifies the lowest-risk group (99.7% payback).
feature_name = 'is_retired'
df_combo[feature_name] = (df_combo['employment_status'] == 'Retired').astype(int)
tier1_features.append(feature_name)


# --- Verification ---
print(f"✅ Created {len(tier1_features)} new Tier 1 features: {tier1_features}")
print("\n--- Sample of Dataframe with New Tier 1 Features ---")

# Display the original column and the new flags to verify correctness.
# We'll filter to show some rows where these flags will be active.
display(df_combo[df_combo['employment_status'].isin(['Unemployed', 'Student', 'Retired'])][['employment_status'] + tier1_features].head(10))

Creating Tier 1 features from 'employment_status'...
✅ Created 3 new Tier 1 features: ['is_unemployed', 'is_student', 'is_retired']

--- Sample of Dataframe with New Tier 1 Features ---


,employment_status,is_unemployed,is_student,is_retired
14,Unemployed,1,0,0
17,Unemployed,1,0,0
22,Unemployed,1,0,0
27,Unemployed,1,0,0
29,Unemployed,1,0,0
36,Unemployed,1,0,0
41,Unemployed,1,0,0
43,Retired,0,0,1
47,Unemployed,1,0,0
55,Retired,0,0,1


In [6]:
# --- Feature Engineering - Tier 2 ("Primary Risk" Features) ---

print("Creating Tier 2 features from 'grade_subgrade' and 'interest_rate'...")

# A list to hold the names of the new features we create in this cell.
tier2_features = []

# --- Feature 4: grade_numerical ---
# Extract the letter grade and map it to a numerical risk score.
feature_name = 'grade_numerical'
grade_map = {'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6, 'G': 7}
# .str[0] extracts the first character (the letter) from the 'grade_subgrade' string.
df_combo[feature_name] = df_combo['grade_subgrade'].str[0].map(grade_map)
tier2_features.append(feature_name)

# --- Feature 5: interest_x_grade ---
# Create an interaction between the interest rate and the numerical grade.
feature_name = 'interest_x_grade'
df_combo[feature_name] = df_combo['interest_rate'] * df_combo['grade_numerical']
tier2_features.append(feature_name)


# --- Verification ---
print(f"✅ Created {len(tier2_features)} new Tier 2 features: {tier2_features}")
print("\n--- Sample of Dataframe with New Tier 2 Features ---")

# Display the original columns and the new features to verify correctness.
display(df_combo[['grade_subgrade', 'interest_rate'] + tier2_features].head())

Creating Tier 2 features from 'grade_subgrade' and 'interest_rate'...
✅ Created 2 new Tier 2 features: ['grade_numerical', 'interest_x_grade']

--- Sample of Dataframe with New Tier 2 Features ---


,grade_subgrade,interest_rate,grade_numerical,interest_x_grade
0,C3,13.67,3,41.01
1,D3,12.92,4,51.68
2,C5,9.76,3,29.28
3,F1,16.10,6,96.60
4,D1,10.21,4,40.84


In [7]:
# --- Feature Engineering - Tier 3 ("Contextual" Features) ---

print("Creating Tier 3 features from 'loan_purpose' and financial ratios...")

# A list to hold the names of the new features we create in this cell.
tier3_features = []

# --- Feature 6: is_home_or_business_loan ---
# Flag for the safest loan purposes based on our analysis.
feature_name = 'is_home_or_business_loan'
df_combo[feature_name] = df_combo['loan_purpose'].isin(['Home', 'Business']).astype(int)
tier3_features.append(feature_name)

# --- Feature 7: is_medical_or_edu_loan ---
# Flag for the riskiest loan purposes based on our analysis.
feature_name = 'is_medical_or_edu_loan'
df_combo[feature_name] = df_combo['loan_purpose'].isin(['Medical', 'Education']).astype(int)
tier3_features.append(feature_name)

# --- Feature 8: loan_to_income_ratio ---
# A fundamental measure of financial burden.
feature_name = 'loan_to_income_ratio'
epsilon = 1e-6 # Add a small constant to prevent division by zero
df_combo[feature_name] = df_combo['loan_amount'] / (df_combo['annual_income'] + epsilon)
tier3_features.append(feature_name)


# --- Verification ---
print(f"✅ Created {len(tier3_features)} new Tier 3 features: {tier3_features}")
print("\n--- Sample of Dataframe with New Tier 3 Features ---")

# Display the original columns and the new features to verify correctness.
display(df_combo[['loan_purpose', 'loan_amount', 'annual_income'] + tier3_features].head())

Creating Tier 3 features from 'loan_purpose' and financial ratios...
✅ Created 3 new Tier 3 features: ['is_home_or_business_loan', 'is_medical_or_edu_loan', 'loan_to_income_ratio']

--- Sample of Dataframe with New Tier 3 Features ---


,loan_purpose,loan_amount,annual_income,is_home_or_business_loan,is_medical_or_edu_loan,loan_to_income_ratio
0,Other,2528.42,29367.99,0,0,0.086094
1,Debt consolidation,4593.10,22108.02,0,0,0.207757
2,Debt consolidation,17005.15,49566.20,0,0,0.343080
3,Debt consolidation,4682.48,46858.25,0,0,0.099929
4,Other,12184.43,25496.70,0,0,0.477883


In [8]:
# --- Cell 7: Feature Engineering - Tier 4 ("Big Four" Interactions) ---

print("Creating Tier 4 features by interacting the top raw features...")

# A list to hold the names of the new features we create in this cell.
tier4_features = []
epsilon = 1e-6 # Epsilon for safe division

# --- Feature 9: financial_stress_index ---
# Pits debt pressure against the ability to handle debt.
feature_name = 'financial_stress_index'
numerator = df_combo['loan_amount'] * df_combo['debt_to_income_ratio']
denominator = df_combo['annual_income'] + df_combo['credit_score']
df_combo[feature_name] = numerator / (denominator + epsilon)
tier4_features.append(feature_name)

# --- Feature 10: income_per_loan_dollar ---
# Measures how many dollars of income support each dollar of the loan.
feature_name = 'income_per_loan_dollar'
df_combo[feature_name] = df_combo['annual_income'] / (df_combo['loan_amount'] + epsilon)
tier4_features.append(feature_name)

# --- Feature 11: credit_score_x_income ---
# Captures the combined effect of creditworthiness and financial capacity.
feature_name = 'credit_score_x_income'
df_combo[feature_name] = df_combo['credit_score'] * df_combo['annual_income']
tier4_features.append(feature_name)

# --- Feature 12: dti_x_credit_score ---
# Normalizes the debt-to-income ratio by the person's credit score.
feature_name = 'dti_x_credit_score'
df_combo[feature_name] = df_combo['debt_to_income_ratio'] / (df_combo['credit_score'] + epsilon)
tier4_features.append(feature_name)


# --- Verification ---
print(f"✅ Created {len(tier4_features)} new Tier 4 features: {tier4_features}")
print("\n--- Sample of Dataframe with New Tier 4 Features ---")

# Display the new features to verify they were created.
display(df_combo[tier4_features].head())


Creating Tier 4 features by interacting the top raw features...
✅ Created 4 new Tier 4 features: ['financial_stress_index', 'income_per_loan_dollar', 'credit_score_x_income', 'dti_x_credit_score']

--- Sample of Dataframe with New Tier 4 Features ---


,financial_stress_index,income_per_loan_dollar,credit_score_x_income,dti_x_credit_score
0,0.007055,11.615155,21614840.64,0.000114
1,0.033523,4.813311,14060700.72,0.000261
2,0.032819,2.914776,34398942.80,0.000140
3,0.006422,10.007144,24975447.25,0.000122
4,0.024684,2.092564,16955305.50,0.000080


# **Feature Importance**

In [9]:
# --- Final Feature Importance Analysis (Numbers Only) ---

import lightgbm as lgb

print("Performing final feature importance analysis...")

# --- 1. Prepare Data for Modeling ---

# Split our combined dataframe back into a training set
train_fi = df_combo[df_combo['source'] == 'train'].copy()

# Define our target variable
y = train_fi['loan_paid_back']

# Define our features by combining the original numerical columns with all our new feature lists.
original_num_cols = train_df.select_dtypes(include=np.number).columns.tolist()
original_num_cols.remove('loan_paid_back')

all_new_features = tier1_features + tier2_features + tier3_features + tier4_features
all_features = original_num_cols + all_new_features
X = train_fi[all_features]

print(f"Training a model on {len(all_features)} features to determine importance...")

# --- 2. Train a Fast LightGBM Model ---
lgbm = lgb.LGBMClassifier(random_state=42,verbose=0)
lgbm.fit(X, y)

# --- 3. Extract and Normalize Feature Importances ---

# Create a dataframe of the feature importances
importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': lgbm.feature_importances_
})

# Normalize to 100% as you requested
total_importance = importance_df['importance'].sum()
importance_df['importance_pct'] = (importance_df['importance'] / total_importance) * 100

# Sort by importance
importance_df = importance_df.sort_values(by='importance', ascending=False).reset_index(drop=True)

print("✅ Final feature importance analysis complete.")

# --- Display the final table ---
print("\n--- Final Feature Importance Rankings ---")
display(importance_df)


Performing final feature importance analysis...
Training a model on 17 features to determine importance...
✅ Final feature importance analysis complete.

--- Final Feature Importance Rankings ---


,feature,importance,importance_pct
0,debt_to_income_ratio,1069,35.633333
1,credit_score,474,15.800000
2,dti_x_credit_score,286,9.533333
3,loan_amount,259,8.633333
4,annual_income,150,5.000000
5,interest_rate,132,4.400000
6,is_retired,129,4.300000
7,is_student,110,3.666667
8,interest_x_grade,108,3.600000
9,is_unemployed,74,2.466667


# **Final Artifacts**

In [10]:
# --- Save Final Artifacts (v2) - Forceful Method ---

print("--- Preparing and saving final datasets (v2) ---")

# --- 1. Split df_combo back into train and test sets ---
train_featured = df_combo[df_combo['source'] == 'train'].copy()
test_featured = df_combo[df_combo['source'] == 'test'].copy()
print("   - Dataframe split back into train and test sets.")

# --- 2. Clean up temporary columns FIRST ---
train_featured.drop(columns=['source'], inplace=True)
test_featured.drop(columns=['source'], inplace=True)
# Make sure 'loan_paid_back' is the correct target name
test_featured.drop(columns=['loan_paid_back'], inplace=True)
print("   - Temporary columns dropped.")

# --- 3. Forcefully Re-attach the original test IDs ---
# Reset the index of BOTH the test_featured dataframe AND the test_ids Series
# This guarantees they both start at 0, 1, 2...
test_featured.reset_index(drop=True, inplace=True)
test_ids_reset = test_ids.reset_index(drop=True)

# Use pd.concat to join them side-by-side. This is the most robust way.
# We are creating a new dataframe with the 'id' column at the front.
test_featured_final = pd.concat([test_ids_reset, test_featured], axis=1)
print("   - Original 'id' column forcefully re-attached to the test set.")


# --- 4. Save the final dataframes to NEW v2 CSV files ---
train_featured.to_csv('train_featured_v2.csv', index=False)
test_featured_final.to_csv('test_featured_v2.csv', index=False) # Saving the new final dataframe
print("   - 'train_featured_v2.csv' saved successfully.")
print("   - 'test_featured_v2.csv' saved successfully.")


# --- Final Verification ---
print("\n--- Final Verification of Saved Data (v2) ---")
print(f"Shape of final test dataframe: {test_featured_final.shape}")
print("Top 5 rows of the new final test dataframe:")
print(test_featured_final.head())
print(f"\nIs there any NaN in 'id' column? -> {test_featured_final['id'].isnull().any()}")
print("\n✅ Notebook 02 complete. v2 artifacts are ready.")


--- Preparing and saving final datasets (v2) ---
   - Dataframe split back into train and test sets.
   - Temporary columns dropped.
   - Original 'id' column forcefully re-attached to the test set.
   - 'train_featured_v2.csv' saved successfully.
   - 'test_featured_v2.csv' saved successfully.

--- Final Verification of Saved Data (v2) ---
Shape of final test dataframe: (254569, 24)
Top 5 rows of the new final test dataframe:
       id  annual_income  debt_to_income_ratio  credit_score  loan_amount  \
0  593994       28781.05                 0.049           626     11461.42   
1  593995       46626.39                 0.093           732     15492.25   
2  593996       54954.89                 0.367           611      3796.41   
3  593997       25644.63                 0.110           671      6574.30   
4  593998       25169.64                 0.081           688     17696.89   

   interest_rate  gender marital_status education_level employment_status  \
0          14.73  Female     

In [11]:
# --- Cell: Final Independent Verification ---
import os 
print("--- Starting Independent Verification of Saved Files ---")

# Define the names of the files we just saved
TRAIN_V2_FILENAME = 'train_featured_v2.csv'
TEST_V2_FILENAME = 'test_featured_v2.csv'

# --- Check 1: Do the files exist? ---
train_exists = os.path.exists(TRAIN_V2_FILENAME)
test_exists = os.path.exists(TEST_V2_FILENAME)

print(f"1. Checking file existence...")
print(f"   - '{TRAIN_V2_FILENAME}' exists: {train_exists}")
print(f"   - '{TEST_V2_FILENAME}' exists: {test_exists}")

if not (train_exists and test_exists):
    print("\n❌ VERIFICATION FAILED: One or both files were not found.")
else:
    print("\n   ✅ Files found successfully.")

    # --- Check 2: Load the files and check their contents ---
    print("\n2. Loading files and verifying contents...")
    try:
        loaded_train_df = pd.read_csv(TRAIN_V2_FILENAME)
        loaded_test_df = pd.read_csv(TEST_V2_FILENAME)

        # --- Verification for Test Data ---
        print("\n--- Verifying Test Data ('test_featured_v2.csv') ---")
        # Check A: Is 'id' column present?
        id_present = 'id' in loaded_test_df.columns
        print(f"   - 'id' column is present: {id_present}")

        # Check B: Are there any nulls in the 'id' column?
        id_has_nulls = loaded_test_df['id'].isnull().any()
        print(f"   - 'id' column has nulls: {id_has_nulls}")

        # Check C: Is the 'id' column the first column?
        id_is_first = loaded_test_df.columns[0] == 'id'
        print(f"   - 'id' column is the first column: {id_is_first}")

        # --- Verification for Train Data ---
        print("\n--- Verifying Train Data ('train_featured_v2.csv') ---")
        # Check D: Is the target column present? (Assuming name is 'loan_paid_back')
        target_present = 'loan_paid_back' in loaded_train_df.columns
        print(f"   - Target ('loan_paid_back') is present: {target_present}")

        # Check E: Is the 'id' column absent? (It should be)
        id_is_absent = 'id' not in loaded_train_df.columns
        print(f"   - 'id' column is NOT present (Correct): {id_is_absent}")

        # --- Final Summary ---
        if id_present and not id_has_nulls and id_is_first and target_present and id_is_absent:
            print("\n\n✅✅✅ ALL VERIFICATIONS PASSED. Notebook 2 is complete and correct. ✅✅✅")
        else:
            print("\n\n❌ VERIFICATION FAILED: Please review the checks above.")

    except Exception as e:
        print(f"\n❌ VERIFICATION FAILED: An error occurred while reading the files: {e}")

--- Starting Independent Verification of Saved Files ---
1. Checking file existence...
   - 'train_featured_v2.csv' exists: True
   - 'test_featured_v2.csv' exists: True

   ✅ Files found successfully.

2. Loading files and verifying contents...

--- Verifying Test Data ('test_featured_v2.csv') ---
   - 'id' column is present: True
   - 'id' column has nulls: False
   - 'id' column is the first column: True

--- Verifying Train Data ('train_featured_v2.csv') ---
   - Target ('loan_paid_back') is present: True
   - 'id' column is NOT present (Correct): True


✅✅✅ ALL VERIFICATIONS PASSED. Notebook 2 is complete and correct. ✅✅✅
